# Downtown Manhattan census migration map

This notebook explores the processed 1960 and 1970 NHGIS tract files. Run `scripts/fetch_ipums_nhgis.py` and `scripts/prepare_census_migration.py` before using it. The census layer measures general residential mobility; artist addresses will be added later as a separate archival point layer.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

In [ ]:
def load_year(year: int) -> gpd.GeoDataFrame:
    path = PROCESSED_DIR / f"soho_migration_{year}.geojson"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. Run the fetch and preparation scripts first."
        )
    return gpd.read_file(path)


tracts = {year: load_year(year) for year in (1960, 1970)}
pd.concat(
    [frame.drop(columns="geometry") for frame in tracts.values()],
    ignore_index=True,
).groupby("year")[["recent_mover_pct", "vacancy_pct", "renter_pct"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8), constrained_layout=True)
values = pd.concat(
    [frame["recent_mover_pct"] for frame in tracts.values()],
    ignore_index=True,
)
vmin, vmax = values.quantile([0.05, 0.95])

for axis, (year, frame) in zip(axes, tracts.items()):
    frame.plot(
        column="recent_mover_pct",
        cmap="Blues",
        edgecolor="#f4f1ea",
        linewidth=0.6,
        vmin=vmin,
        vmax=vmax,
        legend=year == 1970,
        ax=axis,
        missing_kwds={"color": "#e8e5de", "label": "No data"},
    )
    axis.set_title(f"{year}: residents in a different house five years earlier")
    axis.set_axis_off()

plt.show()